In [11]:
from operator import add
from time import sleep
from typing import TypedDict, Annotated
from IPython.display import Image, display
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from datetime import datetime
# ============1.State=================
class ParallelState(TypedDict):
    nodes: Annotated[list[str], add]

# ============2.Node=================
def node_a(state: ParallelState):
    print(f"node_a 接收到了 State：{state}")
    return {"nodes": ['a']}

def node_b(state: ParallelState):
    sleep(1)
    print(f"node_b 接收到了 State：{state}, time: {datetime.now()}")
    return {"nodes": ['b']}

def node_c(state: ParallelState):
    print(f"node_c 接收到了 State：{state}, time: {datetime.now()}")
    return {"nodes": ['c']}

def node_d(state: ParallelState):
    print(f"node_d 接收到了 State：{state}")
    return {"nodes": ['d']}


# ============3.Edge连接Node，形成Graph=================
builder = StateGraph(ParallelState)

builder.add_node("a", node_a)
builder.add_node("b", node_b)
builder.add_node("ac", node_c)
builder.add_node("d", node_d)

builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "ac")
builder.add_edge("b", "d")
builder.add_edge("ac", "d")
builder.add_edge("d", END)

graph = builder.compile()

# display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [12]:
# ============4.调用Graph=================
result = graph.invoke({})

print(result)

node_a 接收到了 State：{}
node_c 接收到了 State：{'nodes': ['a']}, time: 2026-10-19 15:35:53.128519
node_b 接收到了 State：{'nodes': ['a']}, time: 2026-10-19 15:35:54.129248


InvalidUpdateError: At key 'nodes': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE